# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz3DaS3+dX4JhKhYxHzIxkO44ukzrbkhPXybbOcZK9VVQszBCcocVX+Bhp5PL9rf2+v+y6GyAJ8DGSvV5XIpEE0Gg0+o2HLMt6lZZpfswC+GUXDjv4icU8ijg7OmH2L1Wyq3hydHJwyL5jh/GWyktxU1Y5x8dnkUh8kbuTyW9FynzBcrGq8iItjieMfcvepX5asAj+j1Nf4O+/KuHzhK14vuZYJhIGff18/hvj+LAVt8zORB6EEXv19G8vX71h3wEkxp6evXvjTBkvQpakbMN3TBRQjyOEMhesSjgNge0QTBGuqxAKhEtonEFHmQCsWCFYBihi74nq++3TV8xe8dVGIP4ZX4cJQIXHszCpbhwWiR2MMSUscAQ8X23CrUQ94zmPYFzHTAHNARE1Vlby3AdUxLqC5oXE5DRiZc6X/D0UizhLoTnbcgT1nS+23xWbmNl+WKxS/ASYOWoAkkSEfVqKtWCrNMGuVjzxgY7/WU8Es59DvQfUFOrkQBmREOo1qtRSJxFbiwQIuQrTpJ2Nlm65KKqopLmC7tcVDooqIsUl2SSqDEYDE1NskHZRPRst2d5XRSkKpFQmyjDHKnFYxKqiKFiYFCVP4D+RuhPLsiYhEqhEhIGncsDVDSpgPKgKE7QK6vL1qn7C3qNwWb+Gaf30vkiT+jkt6qdc1E/FpirDqHmrlkDmlSiamuUmF9wPk3XzIYybxrdhBuwqJvV7UsXZDlFMskmQpzE7f3nGVNnLmK/FVP56kxUTWcFl7VCDcD2Z/HL69pQtAFU34+XG9cM84bGw63e+LPC37XnYsec5zgTmzDt78/y/oVWDrHuWrq5sZ/L7y9M/foUC24LektKaMmvJV1f4OxIBvefhelNazmQyWUW8KNhpAmIgjmn2fBEwzwuTsPQ8uxBRMEUuDBav0wSGEqXrRZaHSenI2vgPK7kZ9Ij1GLCFHJfri1KsSsDIqAgQoCr8ND/nIl5iAXZjlhQbnonxEi/e7iks9zYcKc44DHC8xLsKEx+KgWeNwvfpsoDPHyy+KlFlWMdsBsQWibdKI45vH80GHiinbSiuodELHhVtX18Bx6G+AEUq/BCF0cYHEKF0G0YOSTfIELAlqKAVqGwQqYCD3JrwRckJnyDNY+z/iYv4KH0O70eP6MNSKhP48Hjm9nAEIB6yn8ag79MwsdUkv/ntnXfy8i3ClSi7KH1WO+llvmtZRUfNrTKfl8LG+sAV3LfTTCS22avjtJDEzUpkJTulX6C/TLAZMLKJulRXQICPXVbj/q5HcypSpPCyNI2gxipw35F4ncP76Q0YOzCedsxvvOs0vxJ5sTgEvSk548JS36zLFmUgR8yvYOryokcwcRMWpZdeLd7llQBZlBN/8C/8k3a2kWLgLiSrdy1Q3AslzKCQ02JhAxskK1Cy32lG3yLu6H2Ot5ajiTvo6jMhWNkY+8ZEokElu0v86Yv3PI8aow82w7U0kSFVuKnWoHnWAV8Jb1Mta8VYgKHzaPo8IF3bBNQfStjF5Th3Aa+zHIxLB4btuDRwszL+CwOWU5kX+thupFoDW2yplouiWwAXjtRt8X0ACBdlbgcuqW/iagIVyO629L24HOVzNC5iQIZAh9rW65T8hMpPwXURZGcb7+aXF8cwpcIxmuYCpD9hs0nzFXkl921CzOwmAWLPjC/XIWgCklOsjqZkCV0sqyAQYBXWi5mD2AZ9mlxvYJAMWX2YXkvoSoqm/fgh+/FHdjhzBivCfCVpyZbjZFcjTAYrJDgfIOX20mkpgMoebbyLPzSDRaMdVgLzxzRScWPiUaYlR81RVLEtbkD4Mzv3p5ITnL4ltAPrXEqM8iiVkwzCcsw+SGDgiX/77dHs2J0HH9nPz7DY6g8tsD5oI2AHrDx2Z8HHQtPEii4EddLaemkMszATmoaYols0ZdBgmWpMIYV2d+SDFymtKDzUUtuojJPw3YsovX7FS1AMyfocQEehZkqxL6DRXfVd7A1tJPjQ8OrbHT5WWAZpBCp7QfhWhfAKHoDjkRQpqGepWjXWoQGZU/YVewFO0Ob3EyC5L1apHwbhisITGRn933x2A4/gG/MsVJHBqPqph+eKhIMx8QIEvfVjO155PFqnCyteWSZjK3F/t8vEaZ5DcNYDONjLaE99sbmHQukrFtaQBUMwiFKyNAmhowGVck/4nwxbMSyOsuVX8hQko16D4wNeHM2yeiMHTf9AE65/+KviUVjuyJ81rdpzFEA9fHXZO2zN7EfoXcD3Bw3eDoYwIsKaKYVCgBo4uxprUUwEfhcrIBBDoW1CQ/YcSIQB3jqHeA4sKY+XEJvJYLmOlQF4A0pFmAmCiwCbmUSn9gMVFs9hXBAdGka2jl9SMNGttsM3F2MCcHYKd1X5HBRVGVeRCwyfXntlcHQI8olE29MoST61+hJ8iw24rFfd2oZakZGAwvwZQFjnKRDurQDnV+S6MOvzycKC/HWT7Ywai8ZTA7y9NPHWWWVdYsCCBqXllrvtAvStBS2DPRtBTW8UJqy2awYMYYYv48BJkohngauYnrtRXCvZj7QWd1zX7eidQa3SCZ2kM65ZiGG3cdQUj/2zNqq5f+CH5cH28GB2QAQABWDq7M/UL5IEdymuARYx+LLLY+bsfNbcyFmZgpiTtA7MixH03nMChggKXynC6bMaRMw4knG2o8j4M0YHgBXvbXkewiyBi1zy4jOYT8Xm9xo+xCefzoGDTBhvvywXtgTZx4mdabENlgPtNDIxTn/WPnPOzNnax5t3T9u/d9KGpqzP5//CbBnzNMXQqirQyBYpGOA5JS9HZk8mjZopka+D82FYu1LcDLnQ59i85zgTxhVlmewLjRbUma5FLztmRQB9IGx2wIMZbmdd9qLoAuPxhPrbQ7dgiKkIe2Z/+IZ6/waRaR4JokToG+nRfPNxgNP2+r2jBb0U3jBB90cWPaYbVrlG6DEcq97t2mtcZMmpUxQ63osSoF2IL0qBwUE6k/3g60wo4Dsc1EPYfPWlwpH7GPP9KVzddWvkE32/voy2+c9CgBBRyhRCy/BWBsnoRMJHPdXX6+DCSoMAoxXr8vgOROsYjhx9b5VVnmraCeZQau7we9Ri3cunJr2YvQINlmOKfgdxCgYY7bqNYxnqDCky5Cn3E20ZMqtt6P9p11zpH5zB9FumTwNp0OaL7o6OhMBumdoWkMxyhrO8hkNHJdc8j6vMHszGvFILlxGYQpmKuTOtUrf2YoHBdxOoqm6w2Iwzf0s4BnDcxyWxJAQfHaI6uZ54zFZpnIWRDEOvRJ7AlONMPf/t5CnbAQOE63Y0lHyN1mkelrFMv66qk9ev5bopwMhywDqv1+Y4Lj1yzNviwmuUru4XLapkjREYGE6JzjpUWRm9OrCSoSBFmWHh8S2H4QFH2h1mkMH+naFXP+E7tOZ2kvNrU+xjgEdFbiKubevtz8+eggqxH80Ppwx+OPA8mzL1n2OKnV+3RbAu/rDDuFPFFVEUZoWw7fn3j6fsIYA5OoKHw9nMofxftLDhecrm3+OPR/Dj8NGjbkfAtSCcyTpCOD9A8/kPCGiOEL9vAT2Gj08QzONBMBhySqr/TIu9mK/0xTZciYWF82A5EO8n4GCCcgPV30m0UtJTNg+TQORoF0gz2dC//M6rMl3xorQluCnzSzBxC1mIiqucP3aGE2NAOSDkIoynuHLqtT0UpciKBcxGuipzIbxcgM9VoX1YzA+fjPiKCGIF/sxVsag1lny1LqdqpRuQWqwBalVmFWhyxNMqUTCKTde5bRm12CWrTQ6q81bYzmTE8cEUDrjKYDRRlO+jKj4ps89q4DEu3KcsjcMy9Gv3U8vfDhko0LrwqM1Bz+71bV5XAgclzfRbXT8MaALLkAxYTstnIneRvOqtFsxX8OktfZmMWEG1QqYPBjUP/LqjheyoaaA4QX7teLZaY4XeQsMMxCTgVVTq3NcF90lhiz6WBU1JHxucve44P3cNaIW5RJ5jUhAYhydk5tWKb5dzapvVsVC9eVfzlxW0a2KgxDQY+G+L6lY2cLdhXqKuga7SfGePqSqZhqQ6HuYVV7wE3eTUyx+UvB+zJDKcmI0JKm5XqcjeftjGblUI31hU+Q6/dtZaZnKt5Z//GFprkestKP4f1gqCYpJvtjmPvfXym0tsbn3S2vWXWAAm5ZqIQl8DzpRCCOM1ru1g/tFbruXSiKEfoMI+xWD60li/kHt4VqjUAXZR5k6Xc9aN2aVFQ/iguZpt6ZuscMVNGHgQjiTgsxbCrLrhMN9RtsH9DFDgojliiwWTdpx8xiRzeZ7zHTW8gIAS7OalCw4WsMePYCRnOuoNGWTTtGx7GBpBm8xF6Kh1tiIvyYuwnO6AujWeamzwFe4cA5EBCQVuS5fvBe1hw8ArR6mVu7wq7ufooePerRj8QtEqZSTA4EibGrtiym4KWS1Jk1uRpzZnP7H5zHDycQn0pujM1w24EzfzKdvB790cYNwUkoJTeuI3+LRrvu3UtwGKERHyNLNthLkjuOwBmxNc+K1R7XrKNqqFoQJxDKCmcbuFjXUc9i2bu/Mnmuf917BTBxQoRh264i9Q/+BsSJ617QIs9TWoGRB9bApvG/nm9BaioCnYBTSx9nx2+BD8L/jpqP1d7tnT18///uZXh0wtEfzwoVRMxV+t0i2yKCy9YiNEWRtqfAYHJcdUgtyzNcWNWlNcPZmqrVpGAHESbkNfboLcpO9xp4VK3clNihAzCNy+h6tDuJ6My1NBmvip6ewHsue9Iq9tncnXS26KwpjkmmpY9iGjSk1jqFGjzpBE0rQElTnOqJwZQoB4NVJAZG/qrWgTD3d5stOd3EKsgUMAEyAiYsDzkjaVqDziVDMkGOSGU0ZbPgQ4m+hPCgylu4ouYNtGlUjIA6Fup0e5KBl2InxFKgluP6jeHiZ9hC7PgJY+MDh2B/04hvwPQR5qhnoCx6u1/oOISrHfxfzSICvSsZDZQ8pj4SdkNKgHglVczHCG/gBBnrmzh5ddfYTVSVkfk6oEZmGbCvRlcYwrnSVuGl2DJyGK3iYVbE0C5JJ82dbUcpze2BA7G1VKCBj8AcY+caakYuAD6qTmq9yfE9L+HIjGhJ04LbK4N5NApS5wb5jZsnaKtbs4tK0g/jC3omEbBAVKR6pdBwHcAjzqYEoYd9gMgFxg6WXj5aJ5RwnQlK3S4fQVlRXQ3nF6OXIl4ADxizgetKrVxiSAFg9zpeAwFoCf/MbDfV7FfbeT2CdCRXDiBbRTK6c4EHzxqxW83NMVf4EhqcgViJYIiBkKkVFsOzZ+79UaRKdXedR73mE98FeNwhgZqSlyMRD5VZQdixqDUPq+h5Vssy717aKOhNi+kJR2JZF7ILAsTNaecn38CpgUfWyvbm7vaYKz6YE34UFIEWLW2xMwZ3uarCKwBN4mBWFFA04PGAIdzmazPf14aY7BrmQUb5VuKLaMulEDRhcx7iqgfeKSMs5gZCIDfbOxnE5V5L6Tv+2Y6ChuYIhQRJ5OTLTUPqh94ouOLH1iWN/M4fBqU43ZQIp7cN1jz1Dr4QKFwYscm2kJWVJDTrUbhDeeXJwpOvw92iCK6unu1R+kz+H+RHZO7gsdU5CJDxm+HvacWI3n2U+tljkemnNNdSjBbfUSWPdFq6O6epJI2zpxlAdF7aVUHO7iRpWNOaw6g4XzKfwpW1eAPLg8ajddL/AaiqHjKipDtBdXx+AJoKm4Ul4I9eSGpYhhGqXfoWXPW/NSryhL20jwkD7z1jkC/0IuJO5J2MOHy2bbnFpF3J/uv3TYn13XqMZFrUJ2UKD9sVp2WQe5p3Oj5/Kym69OxE1pZ+0qheyru9rQzjPuHCRyyRT2ncsaGh8WtFVhqF25r1lALTtqAhkIlRsEWOiY0DsonieO9NnqL+zHBTi6kiaPPm/tJi03Iu8gbNK0g3x/LIOrOQpuZ0XH+IqkGl7NoWoDKzrNZkcqohT2WOpWxFm5kxu/7W5oLsUKkFSzTfQjibpQh2VaPvrEJDrNDTw4nd3DXzyRrlSZkUtfDyfTNQ4adZT62XZsJb86070J9lGYtcbzCsxgL2gcdv3RMRPz44gNJuwhghhSzVM1o1/El1Xp0kbbqxyu4c0qhY/+wucm2hUkM4ILgwYm+48FUKrkZVk70hZmioFJOltX96xVN/jpgkDBi8n1l7gxRH66upT2hlTD2MkxMjxkiQBD+8rZcyDjK/a73F2UC8md+GwHOR1GfIDnOnkEYuu6DpOrluCl4LkKFqWM+2D6RTcEaoIfTukhomIjCUXtF+AzWTwp5fgKzPPZTlt9ChZzcsZ2KXTiCE9g76ncJyQHN+C/3Ym8gWEbosntaF+GuVO1hR05+0rsTAcmzUo9PkNFo05Zot6fawq14RE6bmjSDc2ozh39VMnefReb+mTYlQvsQvrSGa+0BZuw3JXo1V4cH7MfvtcouDHOmPlVnBU2DhHnKS89GL46IjDQkaL+xt2IGz+EYKfEDuaHl60bmFeJIp/M2jGUlvqgJAiMeiSZUc+U2pLP/TSZtzcH2E3eT/ueg+pEam/1Qv6segYmrvEAU7WYHx49NKHUSnpBJwOh34W1qXAZKfQFILChk2SLufsEClNQTWIxg4rm5EPVdGE95+DD8Nyayg9oYOqueZxFLUpA5BJXcQbGpFSZ2r2fiygUW+zxse5KU1S2zjEsi3i89DlDJX3MFCkSD1NHIY8GiE6Y1dHc8aAbRbtIBrWCsTnnvBHrSdeVa4ww7oc4v7Do2TgjSLOlaqoXWVW+6HVpMlVV+Sxr0rNeUS4pYjV8kpU6G5aaeVEV23dZvXnXG6nJq8el3tTI5JuBxqyzjaN109Wk2cBAh2BY/qcKS9pFSJlqS0cTFIk8xkq26ljLfCmxa+TCqc8665VIhPQ6fZdDmje9EQmfCVhKoV6Jvui1Pg5m14c1nR7UScdaXxlQ2fFWKzh1zDfodOvadtjnMMiJRh5U9KQLofEHRvwW3HUI5jyq5HZG23qBh46b4/++UKZPmz5Ut9CXH66UGCyMIHlRx8pSTdFPqafgf6WmGtd62FlsdFYbaKPigv8bhSV/NTpL/pqO0AfVF/3UVVfz1Oov9XsYTKvWlmka2c2rM220miyQL06r3qS3rN6cUeig/WrIy9SZ1rqMvslnzZZd1eoJDb5u6nUOuGo4UR4gpbBNnfgOC1+lc2UhMNCFBZ9A1sdcprf1TRK4KEWN0CG6GjyMagBudQQeqO4eOz+X56w96tyAp5+1xiq9E9ZG5n0onXL8iY7K1i34tr2ngfCTPQcWOYfeh+SjmyVrXIxoWp/WkyG08/pfsbeY8CIhiiC0n7Gv5YExdIP/qgRdL4LbO9C+8ULUt6dwkLY8THPt4BW6mKcX6tA/Ovb26UVz6P9SRRjIdGrXBG4Khxr1LQDY4mFLK68+iYIrNlA0c3+Y4c0mbQffYZ9aA2l66oZNizuxkHCMU8lluLqywateoiZYYYK7oH0GadaZK8uynm55cku6aIkLg6yoYG5iijJ8XF4n9UT3xkjy5WturIcS8eYjuw7bE8zkGQACuAOlwFR9ny1uVAIHBv5oyuzOprA5jhTXtOWI5gO+bWMbOc7eElpxXJu6ofH3qxM61zwsocdHjknBDNx30SdhkHTlFmAYl4mcboFoHQrQJRBtFXkq2i5xh0K5wLmaMnguFiNTNmU+B1OZLPpna0B6aL3Rvke+W+mMIOlUDsDfjKKh8ylIn6K3rKL6JanVqNZc5XHR3NxxiTnKua4r6/tXQNWtrnCbUmc/j+bcHIHrcJrgWjQ/ri8/AplF75ilyJbydKfVyRvVPQyo1y5yBzpynVr1VSSdIYwSN9mqw+uG6iOVqenLLboAxVBYRsZ+TILq9BVla3yVbiUGBTqBqCidMdW0y31X+CyZpEMHsjl4e3RiofxY3dNoFnn42zY3YI0cdOj/I3D//Ic81GV1ktMA5r7oykjlWM+o0mrCnnWEzznhNuAl1Wf6B6auUfUdjVWMK6hmxqyzMM5Coj9uZ8rlVPB19V7kaWEN80FzVkwtF0ve6CzAGOtY4gYXS4YsrsXzVRhF3F1HS+tzGLO1SCNnjmShD9NypWyc5F3d0s3mU2kBiYcbc/eJk2edh0lZ87KeeHr59FMPzRmMVicyuynMAXINnzQZocNQIrPlKlnnTp6q4e6bY6kqVYozvAXncnS6x06IjW5ZvfdSwv22dA5eS7TfTA1pbUO3a04X+mOPTD/sgfqk5HjSZ2zp0tOsmd0brpkGuv3YAodvE81trW+iCypBd/5E9SV1zFbGbsdVwt+8h44OHrkapBO6BgUX2PFYi3Jwj8FlA/VKLMGbm4amuL+ZjKfclA2soAIgt7PhZu+VVcSTRZkHxJfW1//7dfy17339y9evvv7Vs9AvHgkvKCU7El2Uy7FLA5Ksi9AdbK0R57y9YkLmmOWtD3gtRIghgOSYY9ozJSnBklBRh84P7Vb6vimwg2u5icmStMXcPrWTvrkCIddc6yqXo7eX6VFnm7gpMMqWmRu8/IzuGgNvEJ4t7M4HxwKfZbdq55zCrE2kiB6Uj6aHK9AY2co3NtlatgUJIa5Gn1v7MmU3zmTEqKHKAlMPDvz3oKzoHR1wozW54gYm23FMtj1MtnswqcuQWFS/WOsidyYjQpArYkGB+0ZWKVhfigaLY+QNQLXyw5RllYDZ58s8zFuW0WCJpL4/rvZJI16LG26+aOVJXproQkc7Uzfj7YV4zZ3s0d2UcQSOEMqfGXIHckVJt9N7eB/oX1ZZJPpHc7uxtYznm6WIbmCTr4alLugtS9SViKqFDS0HgjudDFCjO87m/Il+k5xbVMsYwrO+k65q4S7EbCpXerS8VZ2wUtkdLR/VZKE6CfM6zdSkk3BiagEnSTEb2LV/0ObJUbk5tMDXfpOKwNwwEfSuXcQzzEqYG0HurgXiHaNKubx+Q9lCTrfD0bWua3RUwECIRLsvh2URHmPZsY5z8RV4l9gkwZ30f395TnzsN3yscW6D7f0m5t81OaMTJGk2lfpj2iHzUt6XGQhXXs2qH4X0llHfs1rq7kKb1NGsup7pqa36MhpfwW0Wz+zTKRu4oxGGcG05n37WRRrFIDX3xYY9K0mUx/Q7bnBIu/dL9gU37GaHVCediyahmmPgUS8W0i4R0jELFafWSVvuhXwhHahpc5+TRw7Ygq73scs6IJqPxIJNKwWxaUcwoVVbQ81TU2MZ3QcsnmOqm5icMZPw+8njQYjqBt+FXC+VlzQvzi8s+UTnK7MKP9D+opaSLbcgTRXDhJlkEpQlzMbhOU+RFLhPgRerMFyonffq+N0CjwVoSwi4u7vPEYo+mG5GyAjfvc5DmEEUGmdoqemHJ5iNiTMOBlRGWKA5NGa6JeYDz/EBs9zbMOvc8Orh9mzq/DYbgD8nF+cMj44b1+41CyDgPC5k8zBb3GZTYruFpNOV2C2uai0ibz5eBNv6A3kEC+kYdNwOCCjCYuNJrWpDyw73D4ZCW6VWqrKvV1pJ3O4XRdlnVyA1kbpD6PaJ8NYZid+2Q0C3zshlkUmAmT/FVuR8UAAB2vbDR6cW+HBLS2ed0ntCBEVBLS4uHcC+xLMToyE698hZgV9gIJteNQhICihVvjD21asyitaF1SgByQuk9ete6rKRVBfQXy5jEjF7VQyh/hIyrQnHhYW8JJFNgv79XM2No/dQAdyS949uRi5w3SgFYf2Z/JlgdnBraIp7CPtn3S0k52NgQ9BoMqAOAShIuuxeY9FxRvMY84TKrw3XSQpTIHCBttDvFlaSFuxZ1Aq2dNIAO/VWgBgunduGftGsfC9DdF9sDPrUq34wxFwUY5dqdHL/WJO+/ReSKVzFotykfnv8RJ857WBzjNodVDwqd3ixzIy7utjd/XuYvYDfNtQgBp+2JS/PvZPTF2dP352eTOmmDlT7EXhx0WJOjHfbX+OnsAe3GwkfORgvFsG1VPKy+/OOQBFJrcdf3715e3pCU+fiZYqIrG1bbpw9xACKYiD8rQIq6baCP9bZh63P2+CIerVvlbD05S7AmyhaCshNmvhmpEtyAa77ShESZUjtX/slLOm2kghPoGNyos4hHfzE+BaT1Bh6u0BRvNEEVzxVWojav3p5dgokeX2Kd9x/aI9FqjDnmF3YFl3cCsSgVQVLgT+mw9C8WSSQlYa9HxvCWXARIlGiY+7iJSEtGPKMCAoF2nW9MUiA/M2uuRubwD3RwYGgraqI4Mkao4AoM5bX5zoR0pE5Pqogr64ZhRLzTP7BCxWAIBi8raQFUxeMkkamB/x66crjeImNT5R6hAidygo4op/PnrG6+N7wrsVScbOLF5xYCAQvvslTeeMplN8fGDgrbrC8IWAd5F48+9s94GShuOVE7e+RDc7pVf75iwoIfh8I9cR/jxP/CiagZYd7jyNKfQTx5DGA+F1ukRWFJAtuqD17c+LcG5jSERAPkMf6VPo9mihajrZmoQLUAcn6HQsMuRLFCkL6YUQGOfhQg9PIkuThMSgqq0lshTkqEgQcx9s6GYyYENI4pLbefnjrEPjr6PGMJqoHrikdASIxrin6u0xx1KKoiPmx3TqqHJf6aobPTTBoCYUB10rbL0nZmAUeCbHkpsh6z2SzTZJ2OnZvRV7FePfRhUqcPzs7fX1yin/r4YBMzcFBwFdlmu8OaPG+yugj5ZJn9HjeX64aSh3WKRkU1WxHjtzBgewgxNkl8sALUMiSdOpChUJobCn6wZskoUUHzG35IsFKmqoC+eIMgWuSMpL29RfaDykbN+8ScL37URaqt0HQzZzhEOf1KrL8pFaSJfkOWhVN2I7tCcPecX4tNc/UVmXn2z6oCA8V1KayucaryeTXnbeZfjqgSpVau0sBheqoU9SuFbSeAF432PmrDVkuLw1U5yndc/LvgdlwAD5M8EIvfHl+St/BidS///ru5M1v7+5aD8U0i9onvKwCOoM6b3FDT9f4GwfosSHC6LRluSux6e0Ls57SIXkW5DwWlO2kNqqt3Moi/TTrBd67M+DoxeTwQiiJQaedGxCZ/af/wIGZQ2iD4XIsl/JaOZYXj9IM47UoSmMPh3ptKxu3KdEuJfcIvGI6keDiRdWZPXdwn9JDur1sQLEOhJGgJMow6UQpNPn1nQJEnFwdmu+n35UOkpfgjnsFOrnHzX1da4D0dUe2o2eTtUWTHb9j3aQLaeSiyR4zXKCScrpz18dQXTthX4ndtN72ZV5BERdj+RGIN9hPC8nZ2BHAqGkxfqeokgO6BmH8RJbGON29aCArtANNE/t0Xd7gEizE2pZU98QMRuY2w7+jgsoPT1+gbpqNbCeV0C4Ojmaz2XH/jIwsnkwm3unrn1++Pq0nZDJBYwv6yhP0J67s7t+vWkfpkkdMNZsotGoovV3QLfjTFl6TQlDI1MD+H6UFXDQ=", "blender_rig.py": "...", "app.py": "eNq9fV1vI8l22Lt+RbkHCzZnyBZJfcyIGsrWzszODLBzV9Ho7jXMJYhid5HsUbO7tz8oaQgGfrGBAEkukDwYMBzYAYLkwX6w4YcEyNv6n8wfiH+CzzlV1V9sUtLe3cxipe7qqlOnTp3PqlMlwzDe+4mIpvwzuxETZr6NuOMGTeYIj4k4SeGlDy9+EgVU5geJmATBNQuYHSwCFvKZ6/O9MJ14rs2hzMf/k4jHwueMM3haihiastRnSeoDCHh+5QWpM/V4JCzDMPbcRRhECZt5wUQ/f4oDXz8HsX6KuO8Ei+xN6KdY2JFIsmpxOgmjwBZxVpLMIwED82dZgbsQe1nHNGjGY3ja25tGwYJZTH2D0UzdmSoUPoxW6E8zkYxlyd7e+W/ef/jIBmxovOILoEhktJjxKogiQU8fuZc6XD8m8ulr7nry6VKEQRwYo72Lyzcf31whoNUeg39PgOCLSST6zAx5HMQtFthJJESL2Rxo3GKJuE3SiDct1u0wqgIzsxA+/B6wRQBTFrCrNJoEBM6gRwOgHbXYwYvDFrTCfy3W6xy+aLZkpUseug7V6kG13tFxix1u1jqHYWCdw44C9WKzzgd+6y6o1hF8Oer2oMMjWe2wc3IM1dZ7b/70/MPFt28+jl+/vwScjX3kH2C4/Xepf5dy/+B1u7fP4xjmd1/c8kXoiTEAnYnY2Nvbc8SUjaeuJ2LTafap1yByRERTIcc/5j7UdwJr5k2Q2KoU2L1SErkzazq5zUusiSd8p1BDUtv9nEGjDrN/uhoIDk9cO7CCyadC66x4kXiFYuCmSMRu4B84VoxfKkBDV3zmscY1FDDH+oVHtut5XL/CAMauPw0slB7gJmw+RUYYQpOQJ3PrU+D6ptNi0yabBhGbMtdX9HKnTFcSt26cxOZmm2YG8hnAnOYgYhAH4ZgowBb+2GhrPDWazWZxYNDfFLUJNgeAINj0NgUZc+IbN5mbpmF5wcxo6W5BwtMIK+tpj9PFgkd3Jg5ZzT2HwRIFQDZNg39K4wT4pMVWa9m5B8JKBMlQmRpPn66yJg0Sq0aLNf640ey31lLMnj5lP/1vLWpsxWVVeG801yy8xY9GAaCIf0yFJ5KAFQDD3GBts1A0T3EuobMOfJAvLUbzy0C/lEAWWoWqUaMN8JrYd+FjLGYpqMh4nAQJ99Q41iwuMNXU+CaIFrzPVo1F6iXuEiabN3A+cjCFD00mvFiwRhdmOwka6xJaZl3XUwRPvRJ2V4ps78/75SE1YrfSraLw2M279bHL2m5U5byjIvBL4CIR9UsTQEWacCWCfPn9P7ErV4BWj/uM0GfbB5bTtDQYZpJubVaHhIXZaGAsRUbaMaqsl+rIvvZoHLWNJ142xvrGK7OxdB0BDNZgz1icRGYNFFmj0WxClQbAMAUIuPzIQo/7AVEbxlQa6EbzbMhGSdaIMR/Csk+fqjmS0l/szCCNboPKBOFWkp9Jt8XDEMhgGud5JeA8GAyqSKmRENKwBGWktFOpGy9wxqQAtnfy7XevoWQmomCjk5IaB9pfIyMvM6XSNEh/XrfYEnWgRCnvcWS5iVjEZrMOsQUHUz+OhOeKpdiO3KWs0GdYH/0uHzkYbCWz0R1Cjy6GUo9rljSyvoyliMBWiXgcu/4YdQ50FwJmMFOGxndLv1PjAnVUH50L9hWTPUC/8DN2Z6lgHH1LrSONsrmrKpdhYwcmjdGa6c8sK22qYSB5bzSuinFAqQVoEIaj5lbsv/z13/2///N7trpZK0DK8hg/+D/4anKpSRMs0eX7t+SuGedpgj6TwfE3MMG7dAGygiVzegKxIJ8wcjmRGj7YKXeiNBQONfgImGZkwc8++KspgFgrg7d0xY2IxvNk4ZnO4DfAty02F+5sngyODztqPOBPv5F+Ozt4zUx0fAX4AyCOPrrg7jTiCwHyHIGcFhgAmC4UCScfH7wR0P/o46BzrhgCjbMDvgI93OMpGCpysBBVMPw5pRUlgcgvHXcJGujOE4OGHER/JX+vw9tTx41B1dz1p564PeUgYH6bxKFvCwxXTjeZxkBb707v2sp91DURQttxI2EnIOl9O/DShX8642G/24GOJty+nkUBaKD+k26ve9B9UQd7Qj5SG8OENO53D6EhAAqi/pMTfuJwfjqFTttT8Py9u358B07Hop26jbMaUMVxU6vY/Sz6h8fhbePsX//2f/2nl/tQ4QxrnX1MJxg2cbK87I6FqRdzVgdzoiEqrMQLcSJE4+yt8EXEI+QEjMzen4OCeqNZ7OX+5Ez1Rj8Vs8eRDfw8NfZlVDSGYGAfHezBqjrNjeI0g87+42QAIpuYGFtZ+AOUl9L+hXnXHBjZg8YKfq4bGvkb10nmMC2dr043OUJOQb9TmZ66qdmc0waoHC+4AYqnnheD8hP+KbuN2jGMx+VeG6JU+xqkrVEBLpvljYBSEv3KzJbZGZVpm5i2HyH6p+ClQpTYToKwf4jz/JIzEMtpNn6IB0E7DRpgwLl/3ahMsVGe2xeTF5Pp9DTnne4BwjyfRG6EQh5yPyFdC/E5xEog01/+8q9e7vN8lpX7HKRJmIL0wkyigmyBUhA6hgqBBTy+mDjAfBC91YQPD4oX6lQ7evq6Kowah1DXlJ1BJAn2g7wI1HUSGGgtQA3fMxVfjEAAkAfIgLxDOFjQOxhtQHDFo0TGFgZ5KWOjSegYMzcKDo47RhaQUHFoTgsg8t43saVKn5FoDroA1mc3lHNIkSkUm8PP4aiGZp9D5SeBPYKWWSRbEpqS3m+2EAuIE/QYFO7WIjw04CNBaFWCIzW3e2rqwyiYgZKPJUhgaLvFFvGsxRIIzB2g7eAbDki1mIiiIFKWBpg64gMDLBX6DndUqrnFTnC1gd+a0H4BROlihI+qgKSQemBP0R1oancGzMxAAkKqqIeY6I+AJVEKioS1ATnZMsEgzwQA+0wBNrvwGZ+bxJdUesY6VucgiyudDGjGOyBPuN7wRBwdvjh0DGxJ45XVTONJd3riHE/oQ97ceHI0OZpMO0pbJrc4dHQa/tt/ZCtqv66A2ispii9/8xfsg1yXQU4NUGRXwutbnSnGFMAEAIyiOiK4AoaPqn+jWUKoDH0F07iWrWtATnHpyWf/fgU0zL4ieCDpBvGho+bedptdVLK9Tq/Xm2gV3YWIOA48EJUnPd5zDo5PKzq6Bzq6qj9D7uDyHBllRlq81qSeVgzcLjVc8iGq3gFoflu0JyK5QVugFPQkSJJg0SdbvAEZWvhnK5jv9ct9es6tbq6Le53MNVjBr3XjbAXCsf4qt7W7MFZWb9Mxed590XUqZAQsTwNwfqdgofpz13GEv5se0rxKfE6zrsDYFroipE+B2fzYJX+JGjHrMN6YsVWjkTEicnouOgXeaNNKXR+9ZR61yaMA6psnHUfMWkqSWk9OJs/BpOn3ZgEhTdXOV4xQpZCNEOPhYsm61lHMJHC0Ya4PTuJpA6he8mvK1vQlkePsT67FHZnymCGoFS7wrgodh4GiQId11klQ+6lNiHXWyBEEtGJiZ+R+JQIUIHBIiyGMFvPEFJ7JMwC9PBdC/xrT/KJ+XcDEjiegkIHtw3zRt2xOp8DBcgVYqvcWLY+P5eMsdR3u24I60mECvECMnQgyISKowJPmJEKYMS2zInCQ/IRPPKFXmhGgjCrHgZ89g9bn10KF/tJ4KNNA6hgGj/qFdAsqZBqtLikEB9wF7plF1hvkJdMg/xdiRbmc7Sk/GOKVaYS+Pbgl5B3Pg08U3NJ6Vdy08iA2o0feB76h5aGNBAt/oZnCZeunTw/QjHSVffJnUC1f4TebtcPp5+R7gkovqGCT9Rui6wEQAZoFaslNxkQEc3P+c6dJ9jSQbWX8SkXaUU8o7pxCqAimDteTMWi8kGGwL2NKmE0olAbcABFVL2tpuzoAoGBj96hUOwjAtDCtBQcoTobQ2UhZevUGDocX8AQqF7w9/LaQNXOYN0F0bRbAJdFd2bnCVojvSJEpSv2fLTj1awqlzoqCpWWJohdx29QClUnRvfBwGYCmiMQNfjS3CV1Bzu7Hcocc3ttYyeZATpB6a0p3S0uwVNjIPveCo5XMAVKIaNdkLwegk1uZMznABzW6eDAJAs8sKIXm/R0E/hg5HeKxgQo9nD7MkzUGEU4w+h+PTUPVAN52Cgwnbm0RJuwN/QLNjFt5osxdet8PfEOB7FT6mJVaYQQjHAM8s7nBnCg/yJy4biqUtIBxHOS7i9YVPZkqoEOWB0Q5sKU/uIpSFcOA+0+hiOriWgiMHExQfGnooLlotprg1R5KKZ8HNz58JqVKJTdzcPERiBuPIcpcCrMc6ADJSFdoUjUzJ5hglcnypOSMAr1B404iwe44Oqb5JjC49YK2dbGGXEp3P4Mv6YFmljNcJpfCGgdd+nDnCs9hZimmQdpqZEc49mebxDhgz+oYyDSePi0O4OlT9uXP/ytiH6bCAdsODpKHayfwbMOkcHyTi15vbpEheGRtWYE0rkSEO7m47J2P8suf/0/gvSJ2tY3LEVamK5VixECruB2GQtivIRNxRoUYpb4f1E+ucVXI2rGOMmuW8XXe/x/YN0aIGEfK2DGDX8BDBhQytiBxUnpfCsdYBoOD+uhvOQUVg4Y0kkwulek0RZWI4W+pGPdboRTd05U0eUGKRjNb+oiGhuNGiDU8YaSMj6TiMFzOiLScZu4LgFoqW4yeb3HtXdINewCylYnTzclC4TXpglYtRTKBRHETM+4EpwX2kwvoPqtuyez9DAR0SD81FGxavFvGw8a0MVLh+vorQ+Mp1Q6qkW0EoAmLPWAcs2t1mntVma/K9PEmniUhqEdarj4oNilLnx7Pv/7tf/4HtmtQFCGrYTQgZMZdJ9wUBLyaP5dP6kf5DEENu/3DEQ4Wnw9H2/XHQyatPH69LFMcMmnCmDbcZRwy4Yk9N9XKEDkoyhXJbDcgdKGezWbuutOOPDa7x0lf8sgF04D+eWzsdKA/u+h2eQF+HI5wHyZbx3NbcilP+OlChU3YdYHBNLqmy/bBG/RVhRZp9wFubNOeO1C6u95f5RX0Po50xMlkT2lDLXZ90BHg4JlTVJqRWoybWj4Ehts91ajgoSI8IuqgzgUkOg/oZ9mfQDroHSfgKVwwHJVrAI0KW1J/8xcsW3+fUHIV6EV8a67Zl7/8LzUfo2EDoDZGpdE/wFEq94urWlv67bOVqOyQydkt7ZMFM82Fi7sxZbzEZr5V9ZZ74EhQwCSzYWLwuXnk4DYeLnzC7MCsiyVUCSrhXzPfncKdIeKnjJec+7NSZGKX9d1vrzDzSKWooHiByxArFV3yq4obX24sl5jLpMPFJdcveDzuAvncBxk1zV0pOGiwYKhjOVQr9GcYvMkyCoGoqHY5/WEL8TCwfA1dbycvZhUXGemo596Ezy22MfVOc9jvHo0siDI98JlNY4yYsiy7RzECgRr2jzsjNfkBAJXTb4pl0kcF8lF4wk5e84QrMgJJaRqdX3IK97bQSs8gOcfcvzPvJ2I2X/eGMrl7+qB5VQlOuDCBNBgChSwXXOzb0V4N7z1601VnRc7TxPXyIJ1eLTsI78qw9AugQlM+pl2J8bi50QcGYRJDMIm4GKHWFYr6Un3CjDQLAlHHRE7YRL6cupbtElTUVcEU8DguMtzmbtaWNDBZhHYb3ru9g8PMSs7RbEZ3Wjt9Rl58LP/Rzk89D2oVCWJxlIkFWLMgjWyR6cQy7eTEhXFp4nQkG0T2PCtcLnChiCpaSzdKUu6NFxB34niySqiMqJllpw634PsYDTRSyqSFgcJHDC6XmKA68dABl3siHcxVy41JnYEv7xisFInQQEfuRJjCCptrtA+X5x/AgCwXVhoLB8w5rr91+lZ3uoaXzYgMQEFdmTyUVcYdjLdfY1bNwgpFhDv+668oO+ntxW9ZGnOH9+thmbNhdwRRxWzYGTWrva/oY7WXUipVDipLlojRKbe5PRcwLvQ9hEVvmf2ttb23G7kRtLpwi0w5fvf+4xWt8SVGn7wlYxam+nEZ8YV+LjzaeQ3QwSE9r7OMSVpP0ry2yV7IIEtkElyQCG/gR8fqbPIlCnSeY01ukOGDk+PydrxwWbv9YyqiuzYgO0DQ7mdasrfgvSV5kiZdP9OktoxdSpWGgj4hRHkEJgxuRGQ5Eb+B3igvLxnY8bLlB3Nwi0UED6nvJrGxc9kpngvPU261zUOErvSIKsS1Nv0IwQ18Gxw0rThx4CkDjNuRQ7nIdittxC2l1UK9yA3NplzvNcErMprAbvq1ZSi1T4QHVxH6CFuS6hzqUW42RCxcMiOuufJhD4tBvfDhwQh/HuYgkB/GcjsSGFy16nZ6h8TW+ytsnJVInjbuUbIFkAa4yihVssm92mbJvZismZ2YyAY4PgA2WOImBfzK5BUID58VKHgcq3JcYMBJHyRhnsx1PW5J2iJ4nR+Xo0vyMrwej7QPc5urJMxryr4P++2TTimpGAG2svHC0wJnQjuuAbBSoBarCr5rxKd45gHEfukuA2aedFjctNhl8ClgdkoLR9yb4RmJWCZWee4CkAbxmfM73H7mmQtbEq96ZDAe1fK79wBfXqdLl/YAS1lLjbMPcmTgXjDcLYU30Pbk1xe3zraEcp+CSSzLeYI6y4dJNrAQDa3B7QSIgrsPuDUBzh+OF99UJva37z8UGeTkueKQk55kkZOOZI2TI8UILw7UtgcEDDKH2zSQIZVWbOU78/CIyxbgfX0PVsbQuhJryJ1GXWOv6q2p6rq2SgDI4b2S/dmqP9HhnYPOVmhXqH+1Hsb6h9PJgXOCjz/94yst/r9rsXcwmmNMmTiRyhY9DOAAew7hVDlOxnApKYc66KO1SDSAwi2WkttMNMp5ASPejPtzNb6kcxJzcoHnwzZojLMBTkypll1NkqBm5A9Ah8WAAj+UIwpCV0sjIloKQNBIzjHRp1tuFdJOmaHzaEGb/Q70IMb8L05ImbVW78B2v4NCTDa5JU3ZJN0mbfhOW6KWGm7LSw0w/JNOf1RJWaKJyKPhl2Hg3eF2MwsBsSQeNFaA67qBayTeoOFDhIXJbFFwDaK2wsyuHZhMDVmzTVvsUL9nHZXJ27WO1439s0LcRxxRwAfzELRo6wyKg/CWnWQJc6Wki8JG/9as21VDzfXBQaOMTkOlczTWWYbDejPnYztgexu4xtkKeWPNXk7OVsSHZKBW6VqmTVC2haICahEMJ0DNSE9ea5bMM4VOqNIZ6+RsVWJEcks9viaHDdsq0L8mcTcJS0huIayuulFR5b2s61JLoY+riE/4J3BFkZIZjRpKF8uTJkhTdGYJLNYjWmxSGozVmHgd/YmX9HTbHTQ6DXbbA279HXD3HbyvUAw7Vhdfe6VXLQhqLEdHuqjt8HjOo4jfDRqH7BBZnHpE1w03wmryWSU1zr789d+xj5Q+7QTVdHuaYlxwNXAsMktKqssfShstrDyzqgep6qGHv/o9uwrAcstUeU2SPzARqjyhEqvq1Ffza2pYTHPhC8yR6qkcKcr/ZL92hhSlUd9EPOzjD8qfpjSpFU7aujZZaovPgTYZAj/l3VCymiyphoKgAvY3ko62BoG4oh/eyNff6Qyt+oyreDmjXPqvg1vgZ9ZhwMxs9Q5YFg/iiWgpzuNQ2Mklhi25Wt+So/y8U5POJhPKkESsA0TSsrReNRqSW8myoDACMg+fI0X2ymxABxos6bDmupLWvbf36iMeUkBvcxLgroElk7tpsjkgFrHVZsIZ+yMZIXIIq/dqWkBI0lb0OOpg3l5e/5SV2HC992TuRGzeZatCrhxyMFOU6qgqIVuVM+t1hR6SkmG12HVEBd1J1+kdnrKtCYisIkxIQ1ZMOUSwCZ/9KnARXWviBZjConqRr+XOpO4o0VBBVwSRfZYm5ckMs0cLUO5JtHtuHzrTabNuoqbTabF4TztLWZb588oEb+qnyufbNmh5J7jBDDrWO4TvCo+jI8A8cmcxhEQWMjBbKcZeM13u8Qn8XJU76ZWZbL03DYIEOVELCYpqmUALzFLDn+wp/JYpKvQLzNQqABXnJnf97kYbiZeuqpDc2gvKldppS13PGaeuDhO3hE4XagMpVPkgYkFraZFFj7H1NY+FGUYuZnCP56kYGK7vuDNM6fJFmkQQblNp7OEGd2u3u+sng2EO+m0QzDzxDZSa8ug/RjZGZj7Qnsbcj9sYQkyNUdOKwYPIOkDtMc75bUyOr6H0BbSt+z52eHRdqJQDQyGogSYlAaHVVdDgdK0qOOKYMXE11iKhyWEVvmaAVJXyIHHpJ4MitZgeX/5Jg9DfK7gQE5chSa2W4ZO4iSfq+yqsoIdpsovotRW2Uz1NksAfxwI0uYP8VTMBdm96QEhW6+7G9D7QGcUfAH8rae25sK9Buyjq1mCvp3x3za3TX2lWzwhbK1XwlvI+EZSsqa5fkOkUbjJHif8a+SA2iREGxvmFTP/AE1u4m43b/yi3A/oJ3cbxAGx5rcR/igeG2RycrZzAhrDWTyzkVsv2eBx/68YQ/jiO2UD0KNUBl4scsSgc4tQoXQY3pvDEYuw6AwOMslHZ5oQqHwAKqHYMzp+wKtI/+BvL/helXehSRz/i4qE6xljMHM4z1NqP/IdHoAAb9/OPqSsiJyiB032/ooOAZmyDNz44oGMrYxmJH/Qwe0mPH813lQBFOFd8UlqE3FLHNNRZPGNL3WLCLzR5j3nzJjHXQOY1ZBvfyBN3ITBL6HpG68F7kYxS8WkPdmBcvv363MhOj/YOOs2tcPQgzm0blKcb+Kbx4V/+e6wSnJkpwNX2HI43F0QqxQRzoJsAHrf6aud0Wx/IEPdUJXECSa4h0xuJSZk+O4bd7e0YdpYOIaZ1U3JBg0Ues37Z/ii9eXuHYL1+wQ5lIv5mZ+9KSeyV/h4FfKxvY4EuXkdBSErjXiIMZaJ7izK+KfmbyILOifwgk8DpC9Z5iBioltgmh5N1gEX0UoVvjO6HveQeemPbcFZE/Q4o4avT8CghxqNkLju2y8xLd5ZJVyVNZQtHSfpfYgBnYnaYiblKmJ4kET+XR9YxaVfaM3X+7+G6heUKUzrwjxvcuUcXYkSCMrsfOjDJiHJsHyEsE5HZsXotdtRiXeu5PDwz6FidbjYFC5FEWYbaQ5Ud3rWA06avW3iEXisk2hOWr5TXYMo9RYWVuqaBhk8ZT1uuZbinlwohYNQ9PBbSsY4zUuSUQNffB9vGncdR4xWgiI0eQQRaVEk2WFBddZVxIfpIGXoXQAvK2jAexYQq32QgHS19l0ueLAvm6kjdkvWMfQOO0fz71x+aFjMe2Qkr3G7VZ0fq5i08BYWcgsevhXzxwGAH90wfTHJFPw4N3KVFlYSXaeFvvC0Lc04lqTBFQJFviGQbDQ9GzccSSlFaX89jhrfNexB9pJXOTuPs4v1/l7oJjzBKde6jk6IV7tLvgqgHJC88eBx3fy8TaM0gxGthuAeTiK4OzeKjHJrCyZcKrupktUJWX9Cgqn64OHw0w5cZ/ze4uWwLedS3jwc41CVwLsb6ciMAj0ckERcsCRzkUY7XbxUu1NmJAGFa4Ve6/C6zJvkFeGqQ6iKef/lnn3bBCcJ9MqHPO+2aaZothuffGR7e3w5xkij9+zXFm5kvXn8vhlHw/mfCR8PofoZxebMdXZAM784WK150R7liMvevOey/GG2FS4e/d/Ma5mLLe/FiE8+C0nJAPBiSOzLKqPXmk4A6W61fKVb54MbqGEK8K16ZjTcoe25jMoj7meZf0i1e7KDbjHsSwFtUmtGd9kCvApuz1MfkCZBljpdb0JmIwpGkx0iKvPYlHvQy9/UIozy63kNmZ4obJdhbgc7H8lQAIPsNxDmUWw4xf+on4FjghWkwAznT/9n7i0Iu80OIfr7kPt4suIvij1DBdEqv5BMc0EZ5i5VNR2ek3IPuz7QgF2RPzS//4X908UCpvAbtHvmW5y5L2HUx8+P4cAO/rsbvoJe70tR8Rx90nrMMXl87KX+X++jpPvByyseQQeHzCq/sQluBDsFD/MuHzJ88qQyVf5MuJjiC3sFhRoGPYoE3Pxp4hkPYLl4fObgnHstOQ++0SOee4EkQubucTn0stkxgvNsTnG/0OZW7eZRB/QaqkOPIuD7IvQO+PDBbgt7uvgC49EPDxwwhbUZd1OaxyxYuaoskQqtt/vSPzV9mKrITu1U/7RDxeY6apNsh/A4PO5mXdjE0soYPiSJ38NeliEGBSRP6EPOpzg2XKHiornIFtNRnQ3N9zlbKvVcnPB/nPH0b4Jn84lGgR7lMk7G+LOceBUvFuCABNs6gxQ8jt3LfFI4gbe/pwSFxtrKkfYP7IP9B7tG90DfM7QWl4ALve0Ei7m0uj10Vl2/va/FYi7dLqDex/15ELI1paRNTAPaZzOd7gNsQ5ZhtGcvPWDYWsS1gZtwHLBg/L64Q45ZundFe6LD/3dWHb81yFmmh/SKom3jaeMxbl08GYkrlmxgzojHJdKo2CgrXrhXAY8sa+OjxFOAXz2PX3NuFJ2nTuEJwowbsjqCKrscAHU1BCAfFw376+3L889P/fZDGeKDmXtLFzxJnwsHcFLh8CfzFDuu5HEN8UQPorQ49doMBVt3QbQrCa3UsHcgD4kPBSILZR88Yj+w5Jm2xGDSP1KUPsiC1MqouK9BXcfAw9O7Gcl1GpmVWbuKASQA9CywFGGtPCeuV4xR9YEHXxmWJpNAVjJvSV5G3Wmo6WpKYrZwoLcVc8s6N0V552ciy59yfCbOIcovp38P6azwScVs4TAlqx7I917429X08dZQc/ip39Pw6N/X8svf11LomNHfP2BC1xwhTi307jcArsO/GlMWO3g1ebyEiciOK37O4ea9uT4DiLXXq2JDXVhXUCKovtVtwhY/lg+T0FVQcTGVRm7bUBTeU61HfQ4uBAU9FNcSjAFYxh7xzpM/MwjnVVn4orKl7Gc5wXYiCwQKTQaEV04FGMz/qqJsgMQv8WOiUTmYDzImWBfJNWtqRGNEnaN6S1nsLpQsnrwqws7NlGg0ymsUjEYtAHsn7+UMnGERaExQLB30zvo9VFvx2TBb++LB0MA5B4QWFjzfcJSPuYRAd7Y0/vrn8/s3lR3kqUf+dABDQMWbtmPgjP8nxyhURxA1gnmAouLzAmS08hxOwgHE/Ad19xyZpTIsSeLdIlOCNIxP1ByyKpywD+1ok+ZEVebzX9TEzC1dyb8Sk6CtgMKZxtcIgNK9rTuo6xdv5+pWba+42LaADHBDEouIRbT/ns3GoMqliZSSp7wvcnMmRw/OC9YglViLvTkk0DkiKEEkRkTZHSrUY0esZy64uzmy7pKElf5mUqRA/YNixNXHx5LLRseg/DI5rfBnFb2Edcb77SNcabMIuHe3WEAB/xVoeT317PpbTTMMbPH9xrAdGAx1s8F8mQfAtTxzL5UoCNSkjNRqj/R0UhqaKqS9JT9m7Oq+GGdZi2z4erngJPxnLi4vGmIKSNQP1Ka9ukQV6t7dbtzBC62fCGVM262BYPQ9bXPFUCkNz1VCLBF6oRKK/qQpIZu3sT8U4WVLdrSj9oZBCFWPHgWloVrzNrHx+EVTUTNAGc/tH+vkdLgPf7tgDNeZJEsb9/f0ZsGw6sexgUcCkiNQ+WF3BYxHvY7penOyj94yKd3/7vpNRaN8GQqW3bb5wjg/J2OLKTeV0O4zVnoPuxlG2WCd4fnRU0q5QXGZV0EP4J2Vi0MwO3eqaAkMNDI4nlTT/dss3gejq5VPYWIJr3/JP8FhJcA3mL428mE+FedL8/ykARZ5nPE3mAxNH1crwrNkkw2rjBbABBHG1+VcPl5U/SDykYSTvXca/vU5nMTGqUoPmoyQyQOnihcglxSgFpSxCxctU7PI53gs6mj8kDsoVvtFu+0Ebb7WXV9bIIuj2IQHJlKQEhKTbe04T1+2vcK7WD1kQk+dsB0UM31+8QYcbXPCoWP7x6jVQtXBet+AzVu44zAyNvEWtfMcUe8mOO5XrUOQRGOxJHfy1kA+88ol3CvfpRmPwAjF0M6NMPwx5+3OnfdIePfvBgunJZ+MH1Bm4fuLKi6Wq8TSI3WLTEskJX1iYyhiaNeEmOCX8unDpV/3ddNrnG9K1Vmifx2ifC+Mcla+uK91ZV2ZKxSsjRaeHnE6le/ZMI/9DYJUzqKh9MzcDh7xxW6iC8NuY1kBwW8jHG0mY/FNktKgk/5iZ/ENlkcxTCyzLMio+cNVZqtdIG1qJlI3UBPW6ZrvqqGX9xxvTnB0IMcIIVW/NFG3oDWUXoLZU/Tnie/8GTSDQKA==", "estudio.html": "eNq1W1tv20iWfvevYJTuFplQ1M1yYsl0NrGdy8CZBHF6egDDSIpkSWJMsjQkJctWC5hfsC8L7O7boh92gX1YYIF9mLd9mPyT/gX7E/Y7VbzpYifdPRMgEVl16pxT536qmIN7nnDT6wnXxmkYHO4c0I8WsGhk13hSOzwYc+YdHoQ8ZZo7ZnHCU7s2TYeNxzUAy+GIhdyuzXx+NRFxWtNcEaU8AtiV76Vj2+Mz3+UN+WL6kZ/6LGgkLgu43SYcqZ8G/PAkSaeeL7Tu8UFTjewcJOk1/fZjIdJFo+GM+vfbnXa3/XjQaExYxAO8O22vs4v3wI94/36HdbzuHl7Tedq/zx/zfc7xFk7xts/2Pcbwxly3f7/n9JxhC2/iEliG+96es9x5sHDEvJH4N3406jsi9njcwMiSZGI6wrtehCwe+VG/NRhzfzRO++1W69uBw9zLUSymkdefsVgnTo2BKwIRZ+/gxhgMIZZ+e3cy15LrJOVhY+qbDTaZBLyhBswzPhJc+/6VmbAoaSQ89ocDMePxMBBX/bHveTxa7txPxWQxEQnkKKI+cxIRTFM+wCiYmcwHAR+m6inOOMSj5yeTgF33hwGfD0Zs0n+MQRb4o6jhg3LSd6EwHg9uGn7k8Xm/NyDIxlUMUPpnuWMlfLRYQbOxa6kSY6Dk1m/TRkXge5qaJQXlk42Yef40UbxNmOeRvLsTxVpnMlfkNGeapiIyLSeNFhVqaQzxTFgMllekDC0X1FtrlB5VCOFZa4OK0ogfjSFoYJrGCVBNhC8lcTWGYBog4/J+JEoZZExZYoWl+x23M+yyDaWvrOlDfMwJuLcQQOun132r28vpenzIpkFKCyYLkm+/jWfPK1Ud84Cl/ozL4cN1oXy9Crbw6HmaFfJoWig4EhEfbBqZsqiWMrb2Nsv/jTawh+fQj1Sw6Lf3aS5k80bmbXut2bj0CDZNhWTeEhMere3ACYR7iVka1JipfpUeVmEGGS1y5JTP04Z0iz650aYPSwCPuyJmUjJKThnzj8muiOPVDdKeKnKSSworvcMEC+b7Y9ryyhbU0BYLzBclIQuCxYZ3QG4s7Us1IpLw+eQWOxKXhZqqOGiYOG5c5fpoAY0jwFK4JSapib70NRmWeiRj8t6hiEPlx7Bp/ke9gRmjEn1WAo0MRZ4fc1eiB0PTMNoWvfLARhajtLq/N7sChyzyw2SxEQRJM6txbvBpmqT+8LqR5bAMc44iN6Df7HeVUJWbSWfND74YotaYsrbxhVS3VZFyXA3cHw6HQOVHQ7FFhSqbIGlJn98jHjeNiiwCSZMrTeda3B3QvnPXbVs9UAn4iEfe4rbIItdnVvNozfd3e+u+X/WqrBBw3V8degrvXdvOSkgs9lbsRfP82XbL2mKgWfUAtFprmSPwF3kEwqKirNgII90KL34kRZvFuPtw6m0y9SMUa4gxK9xt4Wq7zW/qOd98d7njsmjGkvVge9DMKraDxI39SapRXWnX/JDqwpBNaoeL7CWp9Re1dBxzXuvXxmk6SfrNputF1qfE44E/i62Ip81oEjYl0D+0rPZey2o1nakfeGrMCoU3DThW1EyFqglNiihp/jKUfM5C1GBJ81MSNmvLJTYhmccumqr2pcIPb1C05nt2DY6AslXT5IAbsCSxa0jyNTlJVXAipwGgHFPzWMoaMyzk81q+QES1w/dIJtOYHTQV3OE6PEocCOrwrc9vWFJAbUWt4jEB80Qkt2IE8eva4dPY9YOAVTEeNLGZjU1hn1snPE9tNiCitYJKNouEUjv8+V//rJ1Ob7Sf//l/S24qKChJbRNTQF2H7ARqeUuwfTcEOIeZ+iIGZPak6TB94w5RYRV+4FYux7Kn+eOtJLyYhcj0LqsdHuPx80/0fCs0fCEC5Fv60fQYfU/CYi1Vai7ZUjK9U7IuC8Ptgv2/f/vHf9eOPv+EUMJ+sXRdu9YyW7XD51Q78+37AEz7sYQ6QfUbeOxWsH0Jdco8ofk3f7LuEDuAGxVoRDXrVqzdntlGb9lt7t4K0jIft8iMY9+5nbshKvfa4RGrOtiXBY+SaLvcT+YUt6DOO2We41AKWKGX/+RhhIK25G/EIsgE0YwH4uc//8cGnMwShScW45SwN0dV8izjE43JKuEWVy4CVypEsB64aDwW6UrQeorE28Ag23Qaabgp/OoI1hVXADaQXqGYg5+I4PN/jUS0JWARUHLJAzLCP015wFOxHWYM9uAT//QX7YhNVqJpGdXyn5WkpLIHNqwSkvZAY4n2/uW7kxNtGItQq8ssUR/k84s3seOnR0iRMQS1rALlecfNJpsroMg/FSwvTt8/P4XmUb5tQxHIqaRZgq0uP3739OjNV6yvwK0ieCdEeBLN/FhEsNF0KxJezifNtQUK2w62mqTaN3ZiH3rCncoZ6Cm+PoOu3FTEemIMMqjnr05PzuwFwmD/vK7M/MMVd6xR4NTNfIBsFKFhdVBFTv8mn7gwZUoEmonMidlglvxomLJfNkq5DkNMJTs1uMx5+v2roxN7sc5Nv/7i9JmGrMZjoWFQ0xH0XaE91H739uSFsZ1btSgb0XQsM7UXwhOpqb2XdcqnpLISda41dOZY9fzZH+mkTOO5gWv695GfXpva91HMWWBqr32UJgKLd7RsueUEiAVY/Ix+OfKdGjFul5liL/ErhEpgpFrKbsISzidAvnn2O0ClMsuJCtkPsJ6YJyguu56VpAFAz96fatno5/+JtO6xpodhhY1AeO0PvVbGwumb41ysaPRWoDof2r0KVDi9ziHbvW8lDxVd9+uqGNJgzNp4Cm0DV6n0fv01mzCkF02OYa6qfTCNilaj1pNhauZ7XHxAovUjFlvhZBcAf6AxpUuX9lWAjWD/XZSLVTC8//W/JQSk+GFC6Z5Zk2gEgFchQ9AmRuSkpibry8JzYqk+HtsRv1JRx/qBOy9O32Xj+oJFdE7qswRt8pSbJGkezzgM8grdyrPpcIgmh6aWcLQcnYWC/60/58E7Op7QX7N0bGF/ujqCLWfMjmEMqovO0O7ofhTx+IfsmBaPL6VfVdGLaTqZIrKhMTijkzFbsX727sWzcrBEjDDMoZEJGM4gnx6dnD33g9B335dzFU7GMNsrDFs8kidlNu1wsFPEGCrFLazCgqMxWgG9WOohTAWcgIrAk6CR4RUJn9G7bgzkuFU2jxUQuQu9NVfHzAWmSRjzsAL29vW7k9cvgCVmFOtyJqilJnAEUNT2tlxlUYBVlGn9WjjVDRM9yK5hKbflOUGYZZUcQvqETj9m/IiFIKqjSCq11axoi9C1zU6rVfDupoFEtZKUdBAwt8oO4Jnwj9EXkeakCmiYeu53yPwp3zZ2NuHQV9vq5IRHsV8V7Qu8vuTBBMbdbpm7qAbnHd5tq4dOZ7fj5ppBGtJpcbEFdPyi6iqveTLWK+JBvc1fcBHyNL7WOy1s3zAras+MiloEFujVw9elIY/ERGxRRUPdszW3G9Jt3r5qdvI5tAgQvcKjtl4yKkEKTsc89CuMvsRrMhlDyKekHBjWUP7Blrv4s7trWnvVXdPy0nynUQXVcX7+xYI1XB1gmEaWy5J0hUWMKX+y0HqTg5OjQzy7j036p0oXsAXZ2A/vJvv4MWMg264iwCIg2EF20dwpVDUNAjNEuMkemUSR2Isl0rJwL6seR+9wA0fMK6PPxLyLwZewp8cmFYLKchQ2qnwJFZsxX9n2GTZWbEDVQHK8LKP03Dc9yuhyslIjYVaOk4SOuYvkFL+FEej1X3uOEPhO0pQom6MgHTbrxkCxJSlUCEsYEt1wGkkhabKp1i+NBQpYxfKpvch64v55qXXzvGPumt0Ls9QIrMlsWy2T9H9h5u1xtqjjeWbXwrKeCQOgZc7Qe4xlVs+09mmpWkbFetEjq7WstQenAsR5YxeE5eK91qMW0QRGOJL8q9YXLXOF2S5onHfNntlo09rhsNd79Mgk3Vpgut0zhyxIsFY20JV11q553sKynlqlXAfr9iRdtWh5fnkxANPSByh8k4hf8rl+et66UK5Bx6VRAp+3T8/bF3IoPy2TPmFZ1ul558KwwmmQ+pPg+sxlAYv1l0iSfriOtXuhhqtYdyULNLqCuNF58NKkv43uAyAj915Z1rvIfKhScmP40cUTlUL6ZO+bIK8qKPYkZX+o36N1xmILjYfW3nJnWVqYA1/TjQUc1ZpO0C4jD6exP/9BxEiopEQ6J50jalzyk3CSXhe+42x4KNEmPGj3ZkhSXBf24QLMCMtPKEob+cvZJSUpT43Bzy5lJRplDACRgJhDFBf8GaVkKnGIyYGD4ck1UDjlsEEVQHCtuN7FXFjybyxls8lhGSAzypLCF3AXYF9NhOQzjSBM3TGWS6qlkL0hWTGvyHkIT+A6u7HhGzyw2x3p00r6mURdmzCNeHokz1wrSe0Pso2CiA0zyYFklbYNZPDSVrUem+uJdW2iBlCqUSGvOjk3CSCxbowHbWu/KWdSFlFBYA3F7EGe/FDdGsiABRJms5tikg6IuM2D6oAEZWsO4FrzhxIGDYjODAXvikTnxgPPdK3rcjYbuXlYwKzDqy2h6EB0gjSU9ugeA1QjzmLba6Lwka9DenvQUy/Kxt7G4pNKZUqnuqp1CgOsegi1RQlchPJZarfMmd0a3G3mWcljl8Y0mD20RxZLQcuB6SWlYFxYWTpIaVoeqD/JftVEs9u/Y1mzS+ZWaOUrTAOw3+h1eb9TNyxVKr5/fWp//GYhpStLYD1FCSpOBX2VcQbS0Ug3lhoePv8UjaaBSLS//kX7ZjHbAjT7/J8xwj1PDpz48JsFLAxAz5H7Pb2D6c//otHg9bbBm5XB8OND8l2dOLVkc5c8+ajoVoaWqu9LPvbrdTg7S64jVysUx+fotxJ9aCyggUwpsc2umI8qkqfuWB+aC6hnLLx+/eXJ0+N66bxoby6XLiMgbiyyQZljlsuddULJWFzpdNhfydMui7zElscd5zR1MSD7GfoBl3XLAJ2nnhW0mhgqcANWJOsYa8yIb2Mh4YcDJ+bscpmFdhozFEe5PqmcgD7lhYuV3cPYdbriqZfmMcq2ntUe9POU9qETQniwNGIOLaQsMCpYqRs5yr7i+bh5RKmtmE671XrAJW7uNXNky28/KjsFAXiOsVBJDA2RmHE5ssRMWdmtAVQmBmvl31KlHDiPXPEFvxQbdbHYVs8LtGlT1FjhESigjZFqH8hUAnSkTNu263SkVDdkKpC9xFpLcoY46qFwKlsNdYWGSmvfcd09E/IajSOeJH2r1zPp26lAvbV6Slgr1LJjrTpt4qspQgxQnmxjs5ODCtFHSyQrRaks3UkXg/W8FEYysIR+ZF1LCRdR6Lphh1HWGJWDiI/Usa2OAGnZAwzKHmBAliYPpVcCUr2e2QvCH53ASHAr4NEohSIzTMX2n+Ygr2mi2EVu+fLq1D6vv+NgiU6JjtRBDz2JGE0vHs5YMPVY9pTKh2fwRPnwViT8w/v6hcK5wlEi4lTXmekY9qEu6ajw/QaebNG3cMbD/X3j2/39xtqsszKbsVsGBZeCQpWSschkdu7KlRe2FILlBv7kqZyh7Cex5Nt27OKoxEUASXnW1+t1dSpep6qn6t0KM1wC8cPlev0D9q5JKBGBDlo1Hduk4KIrUGNFe9WjGCervSRwznmugIsn+VO/ukWU6BlWtbUsG9MhhsRGARSB8QwhIPPJ29y9WmDmK7LjMWKYggj4zb/Asu9lMNVJectxigSSR6G6FBh5C09/QCOsEzDdXawA07E/86NEgRcJV93bbITode/+8cd8RJ5t15/U5WV6vV+nDw7qG9hKf1nH9KR+4BT3v4eUjY+YxzQZgzRbm0Yqc1o0cyxgk9iWFvIb7CQytYBp9MkQo0FPOAGz6n2pznX2iEh+Iy2p1B+qlK3OaX/8EXn5YV1TbybIssppbV85uaqM0RPLXhcqPVUnjVndNbg9wRVC2X7t8DQIsFRewmu5xaOsjU8YsrpjHzoVvaViNAqUkk3HomtDqNmaYa8yr8uKMDvQoGCjolhZI0ozj/LKD+VxZvHRxYBSNsvztUqAhOG777KHe7bNjOwZparH30zRhHaNAbPomDfVDTn6KqJBSxIyBjkb7Eu7r34U9LW7r4YE24424oFhVKvjzBtEJJtIua07UrD+NI7ZNV7lr14mM+NJ+dw/L58vSqZD+zC0yOOkxdigiPy1LA6apNnY9ex0pF45KN6US/7R5LpI8jhH1RCHuYkJOoUJG8kAVR4b2Y6lPjnNImo2LmxveyhAXFSx426WKtzM7cP5thgkMRnSqIRRpUbZO5umpF4QwvjJDA8ERGfUel3usG5SKP8bsWPcKetz9UXExW2yJkYWhf5K5wvWI8Lya+i4X6CzenSt0ghFmFikd0Z9n1J2zppLMZC+KUA99otb+Aw0tTcn0SGb6rRljtrq5QOr9Yi64LyYWW+oU4CnaOBTNMoA7vVUF1vtiNO1xlbFoWXRFJwzE6VEZWcWYquPEsGENEI20X8/DR0qvLPzC5N/SQ13BtyqLmTXVIm1ZEalJu7S2r3V9636q4a0VXBjqei4abxGJ89EO5XUvsqHcvPIvvcVyX+wFabKFyLYoBI/M76yAuQOwhslyhbCGzDrhJVNU9iu9lTlRUl2KJfd0siSugQswrPl8Uk6fo+gm/nSNhj6PlH9949O9Z6h0tYp+/57Zo5FaFX+q4Bq9EIru/6x5fWPzCbZceHXNqCDvzfTOYftweoG1FmE5Dm3m/Gm4xS3eupBl9syEUjyKMRu7RFYnUqQMbKtveVuEOZ0DMf9/t2pXvfpgrtJl920Am10ROUaOmT1EY68BseEZEsnK9/MSfTpwA1XSWlBcY7JO057+73m3Qd5X3uVvX5XXjSRp0JM9IrbeeiMqAynuH7M0RzqWYfy3XcrjYq3GWxvkz7R3tHlMZIkBLOjMyp5maVOafTqOVU99kcfZF39KZF9hfyFLIsTqjKio24vrsHeOCQca4belieqMGexO/ZnAqX5Yqn6k7WTqN/4WY5Z/USj+j3G6vcXFwb4VHem25jIGL/k14lOn+cYa+BKOFB+6CeIJsg755ZFvxcyaWXnc8obFWxxGGioEzaFKxPBr01nZQepDvoq2ezCSgSSCXhYOdGrHNeGNnlt9n1g/euksHFA+Au8eFh1Tnopu0dCfj68ePhRO5D/V+Twm8UwLwQsbHoChzDoY2g5+RGhqNrrMxk213iXbBK/9zLkv43Xak8y3EpfqVkWFXVAk0CXBiXz4htu9Kj08fZBU/3nxv8H1J+7UQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()